# Verified fracture detector — lightweight retraining

This compact Kaggle notebook fixes the old shoulder bias. It trains **only six verified anatomy classes** from BoneFractureYolo8 and uses FracAtlas **non-fractured** X-rays only as background examples. It intentionally does not use FracAtlas fractured images because that archive has no per-image anatomy labels.

Before running, add these Kaggle inputs:

- `archive` containing `BoneFractureYolo8/`
- `fracatlasdataset` containing `FracAtlas/` (optional but recommended for background images)

The run uses YOLOv8s, 640 px images, and 60 epochs so it is substantially lighter than the previous 800 px / large-model training. **Before running, open Kaggle Settings and set Accelerator to GPU; this notebook intentionally refuses CPU training because it would take far too long.**

In [ ]:
%pip install -q "ultralytics>=8.2.0"

import random
import shutil
from collections import Counter
from pathlib import Path

import torch
from PIL import ImageFile
import yaml
from ultralytics import YOLO

# One source JPEG has a few missing trailing bytes. Its pixel data is still
# usable; allow Pillow (used by Ultralytics prediction) to read it.
ImageFile.LOAD_TRUNCATED_IMAGES = True

if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Kaggle, open Settings → Accelerator → GPU, then restart and run all cells.')

DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE))
SEED = 42
random.seed(SEED)
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
DATASET = WORK / 'verified_fracture_data'

# These are the only labels the new checkpoint is allowed to contain.
CLASSES = [
    'elbow_fracture', 'finger_fracture', 'forearm_fracture',
    'humerus_fracture', 'shoulder_fracture', 'wrist_fracture',
]
# BoneFractureYolo8: merge its two humerus labels into one verified class.
BONE8_REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 4, 6: 5}

def find_dataset(dirname, required_child):
    matches = [p for p in INPUT.rglob(dirname) if p.is_dir() and (p / required_child).exists()]
    if not matches:
        return None
    return min(matches, key=lambda p: len(p.parts))

BONE8 = find_dataset('BoneFractureYolo8', 'train')
FRACATLAS = find_dataset('FracAtlas', 'images')
assert BONE8, 'BoneFractureYolo8 was not found. Add the archive input, then restart the notebook.'
print('BoneFractureYolo8:', BONE8)
print('FracAtlas:', FRACATLAS or 'not supplied — continuing without extra background images')

In [ ]:
# Rebuild the compact training set. Each source image receives a prefixed name
# to prevent collisions, and each image has a matching label file.
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ('train', 'val', 'test'):
    (DATASET / split / 'images').mkdir(parents=True)
    (DATASET / split / 'labels').mkdir(parents=True)

counts = Counter()

def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

def copy_sample(image, lines, split, prefix):
    stem = f'{prefix}_{image.stem}'
    shutil.copy2(image, DATASET / split / 'images' / f'{stem}{image.suffix.lower()}')
    (DATASET / split / 'labels' / f'{stem}.txt').write_text('\n'.join(lines))
    for line in lines:
        counts[CLASSES[int(line.split()[0])]] += 1

def remap_labels(label_path):
    if not label_path.exists():
        return []
    remapped = []
    for raw in label_path.read_text().splitlines():
        parts = raw.split()
        if len(parts) < 5 or not parts[0].isdigit():
            continue
        source_class = int(parts[0])
        if source_class not in BONE8_REMAP:
            continue
        try:
            values = [float(value) for value in parts[1:]]
        except ValueError:
            continue
        if len(values) == 4:
            # Standard YOLO detection label: x_center, y_center, width, height.
            x_center, y_center, width, height = values
        elif len(values) >= 6 and len(values) % 2 == 0:
            # BoneFractureYolo8 stores many labels as segmentation polygons.
            # This lightweight detector needs a box, so wrap each polygon.
            xs, ys = values[0::2], values[1::2]
            x_min, x_max = max(0.0, min(xs)), min(1.0, max(xs))
            y_min, y_max = max(0.0, min(ys)), min(1.0, max(ys))
            x_center, y_center = (x_min + x_max) / 2, (y_min + y_max) / 2
            width, height = x_max - x_min, y_max - y_min
        else:
            continue
        if width > 0 and height > 0:
            remapped.append(f'{BONE8_REMAP[source_class]} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    return remapped

for source_split, target_split in {'train': 'train', 'valid': 'val', 'test': 'test'}.items():
    image_dir = BONE8 / source_split / 'images'
    label_dir = BONE8 / source_split / 'labels'
    for image in image_files(image_dir):
        copy_sample(image, remap_labels(label_dir / f'{image.stem}.txt'), target_split, 'b8')

# FracAtlas fractured images have no verified anatomy in this archive, so never
# assign them a region. Non-fractured images are safe background examples.
if FRACATLAS and (FRACATLAS / 'images' / 'Non_fractured').is_dir():
    backgrounds = image_files(FRACATLAS / 'images' / 'Non_fractured')
    random.shuffle(backgrounds)
    train_end, val_end = int(.70 * len(backgrounds)), int(.85 * len(backgrounds))
    for split, images in (
        ('train', backgrounds[:train_end]),
        ('val', backgrounds[train_end:val_end]),
        ('test', backgrounds[val_end:]),
    ):
        for image in images:
            copy_sample(image, [], split, 'fa_normal')
    print(f'Added {len(backgrounds)} verified non-fractured background images.')

data_yaml = WORK / 'verified_fracture.yaml'
config = {
    'path': str(DATASET), 'train': 'train/images', 'val': 'val/images',
    'test': 'test/images', 'nc': len(CLASSES), 'names': CLASSES,
}
data_yaml.write_text(yaml.safe_dump(config, sort_keys=False))

print('\nAnnotations per class:')
for class_name in CLASSES:
    print(f'  {class_name:20} {counts[class_name]:5}')
assert all(counts[name] > 0 for name in CLASSES), 'A verified class has no labels; stop and check the input dataset.'
for split in ('train', 'val', 'test'):
    print(f"{split}: {len(image_files(DATASET / split / 'images'))} images")
print('YAML:', data_yaml)

In [ ]:
# Lightweight training: YOLOv8s is much smaller than YOLOv8l and 640px is
# sufficient for a fast baseline. Do not change CLASSES or BONE8_REMAP here.
model = YOLO('yolov8s.pt')
results = model.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(WORK / 'runs'),
    name='fracture_verified_light',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=15,
    seed=SEED,
    mosaic=0.5,
    close_mosaic=10,
    plots=True,
)
best_path = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best checkpoint:', best_path)

In [ ]:
# Evaluate once and publish a predictable output filename for download.
best = YOLO(str(best_path))
trained_classes = [best.names[i] for i in sorted(best.names)]
assert trained_classes == CLASSES, f'Unexpected checkpoint labels: {trained_classes}'
metrics = best.val(data=str(data_yaml), split='test', imgsz=640, batch=8, device=DEVICE, plots=True)
print(f'mAP@50: {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
for name, ap in zip(CLASSES, metrics.box.ap50):
    print(f'{name:20} AP@50={ap:.4f}')

# Release the trained weights before the extra safety audit. This preserves the
# checkpoint even if Kaggle stops during a later audit step. Do not deploy it
# until both safety gates below pass.
published = WORK / 'best_verified_light.pt'
shutil.copy2(best_path, published)
print('Checkpoint copied to:', published)
del metrics
torch.cuda.empty_cache()

# Deployment gate: reject a checkpoint that remains shoulder-dominant on the
# held-out test set. This uses the app's 0.45 inference threshold.
test_images = image_files(DATASET / 'test' / 'images')
ground_truth = Counter()
for label_file in (DATASET / 'test' / 'labels').glob('*.txt'):
    for row in label_file.read_text().splitlines():
        if row.strip():
            ground_truth[CLASSES[int(row.split()[0])]] += 1
def predict_one_at_a_time(paths):
    # Passing every test image at once exhausted the Kaggle T4 GPU. One image
    # at a time keeps the audit memory bounded and still evaluates all images.
    for path in paths:
        yield best.predict(str(path), imgsz=640, conf=0.45, iou=0.45, max_det=3, device=DEVICE, verbose=False)[0]

predictions = predict_one_at_a_time(test_images)
predicted = Counter()
for prediction in predictions:
    for box in prediction.boxes:
        predicted[CLASSES[int(box.cls[0])]] += 1
true_total, predicted_total = sum(ground_truth.values()), sum(predicted.values())
true_shoulder_rate = ground_truth['shoulder_fracture'] / max(true_total, 1)
predicted_shoulder_rate = predicted['shoulder_fracture'] / max(predicted_total, 1)
print(f'Expected shoulder share: {true_shoulder_rate:.1%}; predicted shoulder share: {predicted_shoulder_rate:.1%}')
if predicted_total and predicted_shoulder_rate >= max(0.50, true_shoulder_rate + 0.20):
    raise RuntimeError('Shoulder-bias gate failed. Do not deploy this checkpoint; train longer or inspect labels.')

backgrounds = [path for path in test_images if path.name.startswith('fa_normal_')]
if backgrounds:
    false_alert_rate = sum(len(result.boxes) > 0 for result in predict_one_at_a_time(backgrounds)) / len(backgrounds)
    print(f'Background false-alert rate at 45% confidence: {false_alert_rate:.1%}')
    if false_alert_rate > 0.15:
        raise RuntimeError('False-alert gate failed. Do not deploy this checkpoint at the current threshold.')

print('Download this file from Kaggle Output:', published)
print('Then set FRACTURE_MODEL_PATH to this file before starting Streamlit.')